<a href="https://colab.research.google.com/github/AbdAlRahman-Odeh-99/Two_Phases_Simulation/blob/main/multiclass_supervised_GMMbandit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Placeholder for `mydataset.py`

We will create a placeholder with a `generate_data` function that returns arrays with the expected shapes. This allows the rest of the simulation code to run.

In [8]:
import numpy as np
from sklearn.datasets import make_blobs

def generate_data(nsamples=1000,nclasses=2,nviews=10,seed=42,snrdb=9):
    sval = 10**(snrdb/20)
    rng = np.random.default_rng(seed=seed)
    mm = rng.random(size=(nclasses,nviews)) * sval # symmetric
    # shared variances
    # FIXME: currently simplify to unit variances, only means differ
    stds = 1.0 # simplification
    #mm = np.concatenate([-mm, mm],axis=0) #
    data = make_blobs(nsamples,n_features=nviews,centers=mm,cluster_std=stds,random_state=seed)
    return data[0], mm, data[1] #(observations, means, labels)


Overwriting mydataset.py


# Multi-Class GMM Supervised Simulation

## Imports and Setup

This section imports all necessary libraries for the simulation, including `numpy` for numerical operations, `pandas` for data handling, `numba` for performance optimization, and `mydataset` for data generation.

In [9]:
import os
import sys
import time
import pandas as pd
import numpy as np
from numba import jit
from types import SimpleNamespace

## Define Simulation Parameters

Instead of `argparse` which is typically used for command-line arguments, this section defines the simulation parameters directly as a `SimpleNamespace` object. This makes it easy to adjust parameters within the Colab environment.

In [10]:
# Define simulation parameters
args = SimpleNamespace(
    seed=42,
    num_views=20,
    num_classes=4,
    num_rounds=2000,
    num_trials=20,
    budget_ratio=0.7,
    output="output_multiclass_gmm_supervised.csv"
)

## Core Helper Functions

This block defines several helper functions crucial for the simulation:
-   `multiclass_risk`: Calculates the multiclass risk based on Bhattacharyya error rates.
-   `bhattacharyya_error_rate`: Computes Bhattacharyya error rates from squared mean differences.
-   `pred_linear_cla`: Performs linear classification based on observed data and class means.

These functions are decorated with `@jit` from `numba` to Just-In-Time compile them for improved performance.

In [11]:
@jit
def multiclass_risk(diff_mean_sq_mat):
    nc = diff_mean_sq_mat.shape[1]
    err_rate = bhattacharyya_error_rate(diff_mean_sq_mat) # (nc, nc) error rates NOTE: diagonal are zeros
    denom = 1.0/nc/(nc-1)
    return 0.5 * denom * np.sum(err_rate) # only half of the total sum as we want off-diagonal terms

@jit
def bhattacharyya_error_rate(diff_mean_sq_mat):
    d_norm = np.sum(diff_mean_sq_mat,axis=0) # (nc, nc)
    acc = np.exp(-d_norm)
    return np.maximum(1.0 - acc,0.0)

@jit
def pred_linear_cla(x_observe,class_means):
    mean_tr = class_means.T # (v,nc)
    diff_mean_sq = mean_tr[:,:,None] - mean_tr[:,None,:] # (sel, nc, nc)
    pairwise_mean_avg = 0.5 * (mean_tr[:,:,None] + mean_tr[:,None,:])
    inner_prod = np.sum(
        (x_observe[:,None,None] - pairwise_mean_avg) * diff_mean_sq,
        axis=0) > 0 # bool(nc,nc)
    # mask the diagonal
    np.fill_diagonal(inner_prod,False)
    # prediction
    y_pred = np.argmax(np.sum(inner_prod,axis=1))
    return y_pred

## Greedy Oracle for View Selection

This function implements a greedy oracle algorithm to select views based on maximizing a margin gain relative to their cost, while staying within a budget. It considers both free views and paid views, and includes a "giant item check" for potentially better single-item selections.

In [12]:
def greedy_oracle(diff_mean_sq,costs, omd_lambda, remain_budget, free_indices):
    """
        diff_mean_sq: shape (nview, nc, nc)
    """
    # Bhattacharyya distance
    # err probability upper bound
    nview = len(costs)
    current_cost = 0.0
    current_objective = 0.0
    sel_set = [] # a copy
    avail_elements = set(range(nview))
    # harvest from free views first
    for i in list(free_indices):
        tmp_select = sel_set + [i]
        margin_gain = multiclass_risk(diff_mean_sq[np.array(tmp_select)]) - current_objective
        if margin_gain > 0:
            sel_set.append(i)
            current_objective += margin_gain
            avail_elements.remove(i)
    copy_set = list(sel_set)
    while avail_elements and current_cost < remain_budget:
        best_margin = -1
        best_add = None
        for i in avail_elements:
            cost_i = costs[i]
            if current_cost + cost_i <= remain_budget:
                tmp_select = sel_set + [i]
                margin_gain = multiclass_risk(diff_mean_sq[np.array(tmp_select)]) - current_objective
                # now there is no zero cost elements, add epsilon for safety
                gain_ratio = margin_gain / (cost_i + 1e-9)
                # only add it if the margin is greater than the shadow price
                if gain_ratio > omd_lambda and gain_ratio > best_margin:
                    best_margin = gain_ratio
                    best_add = i
        if best_add is not None:
            sel_set.append(best_add)
            current_cost += costs[best_add]
            current_objective = multiclass_risk(diff_mean_sq[np.array(sel_set)])
            avail_elements.remove(best_add)
        else:
            break
    # Giant item check
    greedy_solution_reward = current_objective
    greedy_solution_indices = list(sel_set)
    best_single_item = None
    best_giant_reward = -np.inf
    for i in range(nview):
        if i in copy_set:
            continue
        cost_i = costs[i]
        if 0 < cost_i <= remain_budget:
            tmp_giant_indices = copy_set + [i]
            reward_with_giant = multiclass_risk(diff_mean_sq[np.array(tmp_giant_indices)])
            if reward_with_giant > best_giant_reward:
                best_giant_reward = reward_with_giant
                best_single_item = i

    # compare against the giant item
    if best_single_item is not None and best_giant_reward > greedy_solution_reward:
        final_indices = free_indices + [best_single_item]
    else:
        final_indices = greedy_solution_indices
    # convert to boolean vector
    return np.array([idx in final_indices for idx in range(nview)]).astype("bool")

## Full Feedback Simulation Function

This function `sim_full_feedback` performs a single simulation run. It initializes weights, iteratively selects views using the `greedy_oracle`, updates model estimates based on observed data, and adjusts an Online Mirror Descent (OMD) lambda parameter to manage the budget. It returns the total reward, spending, and average accuracy.

In [13]:
def sim_full_feedback(xdata,costs,num_rounds,budget_ratio,means,y_labels,seed)-> dict:
    rng = np.random.default_rng(seed=seed)
    nclasses = means.shape[0] # (nc, xdim)
    nviews = means.shape[1]
    # init weights
    regular_scale = 1.0
    m_est = np.zeros((nclasses,nviews))
    covs_est = np.stack([regular_scale * np.eye(nviews) for _ in range(nclasses)],axis=0)
    ncnt_mat = np.ones((nclasses,nviews,nviews)) # for stability
    # optimization
    free_indices = np.array([idx for idx in range(nviews) if costs[idx] == 0])

    omd_lambda = 0
    lambda_max = 20.0
    alpha_ucb = 2.0
    step_size= 0.08
    remain_budget = num_rounds * budget_ratio # initialization

    # recording
    record_acc = np.zeros((num_rounds))
    # assertion
    assert remain_budget > np.sum(costs) * nclasses # least budget to cover full samples
    for t in range(num_rounds):
        if t<nclasses:
            subset = np.ones((nviews)).astype("bool")
            is_init = True
        else:
            is_init = False
            # optimistic estimate
            # estimate the statistics based on partial counters and partial sums
            tmp_cnt = np.diagonal(ncnt_mat,axis1=1,axis2=2) # (nc, nview)
            mean_est =  m_est / tmp_cnt # (nc,nviews)
            # print the mse, per class
            # deubgging
            """
            mse_classes = []
            for c in range(nclasses):
                mse_classes.append(np.mean(np.square(mean_est[c] - means[c])))
            print(f"MSEs per class: {",".join([f"{item:.4f}" for item in mse_classes])}")
            """
            #cov_est = covs_est / ncnt_mat # FIXME: this needs to be corrected for the bias
            w_diff = mean_est.T[:,:,None] - mean_est.T[:,None,:] # (views, nc, nc)
            diff_mean_sq = np.square(w_diff) # (views, nc, nc)
            #TODO: add the ucb bonus on square difference...
            # using naive approach for trial
            # we need to compute the bonus for different counter, classes and views
            inv_cnt_sqrt = np.sqrt(1./tmp_cnt).T #(nv, nc)
            bonus_mat = inv_cnt_sqrt[:,:,None] + inv_cnt_sqrt[:,None,:]
            bonus_mat *= np.sqrt(alpha_ucb * np.log(t+1))
            optimistic_diff_mean_sq = diff_mean_sq + bonus_mat
            subset = greedy_oracle(optimistic_diff_mean_sq,costs,omd_lambda,remain_budget,free_indices)
        # checking the budget
        inst_cost = np.sum(costs[subset])
        if remain_budget >= inst_cost:
            remain_budget -= inst_cost
            #valid_step = True
        else:
            # override with free indices
            subset = np.zeros((nviews)).astype("bool")
            subset[free_indices] = True
            #valid_step = False
        # observe
        if not is_init:
            x_obs = xdata[t,subset]
            sel_mean = mean_est[:,subset] # (nc, sel)
            y_pred = pred_linear_cla(x_obs,sel_mean)
        else:
            y_pred = rng.integers(nclasses)
        # for reward and update (external)
        y_true = y_labels[t]
        reward = y_pred == y_true
        #if not valid_step:
        #  this is the overspending case, make prediction with free views
        #  and move on without update
        #    continue
        # update weights
        x_masked = xdata[t] * subset.astype("float64")
        m_est[y_true] += x_masked
        covs_est[y_true] += np.outer(x_masked,x_masked)
        ncnt_mat[y_true] += np.outer(subset.astype("int"),subset.astype("int"))
        # update the omd
        raw_lambda = omd_lambda + step_size * (inst_cost - budget_ratio)
        omd_lambda = max(0, min(lambda_max, raw_lambda))
        # record results
        record_acc[t] = reward
    # collecting results
    spending = budget_ratio * num_rounds - remain_budget
    reward = np.sum(record_acc)
    return {"reward":reward,"spending":spending,"avg_acc":np.mean(record_acc),"record_acc":record_acc}

## Run the Main Simulation

This section executes the main simulation loop for a specified number of trials. For each trial, it generates data, calculates costs, runs the `sim_full_feedback` function, and records the results (reward, spending, average accuracy). Finally, it saves the accumulated results to a CSV file and prints the total simulation time.

In [14]:
def main(args):
    rng = np.random.default_rng(seed=args.seed)
    # recording
    result_pd = {"trial":[], "reward":[], "spending":[], "avg_acc":[]}
    # simulation
    time_start = time.perf_counter()
    for t in range(args.num_trials):

        x_data, means, y_labels = generate_data(args.num_rounds,
                                                args.num_classes,
                                                args.num_views,
                                                args.seed,
                                                snrdb=9)
        cost_vec = rng.random(size=(args.num_views,))
        cost_vec[0] = 0.0 # adding one free view
        cost_vec /= np.sum(cost_vec)
        trial_start = time.perf_counter()
        sim_dict = sim_full_feedback(x_data,cost_vec,args.num_rounds,args.budget_ratio,means,y_labels,args.seed)
        print(f"Trial {t + 1}/{args.num_trials}... (Time elapsed: {time.perf_counter() - trial_start:.3f}s)")
        # accumulating
        result_pd['trial'].append(t)
        result_pd['reward'].append(sim_dict['reward'])
        result_pd['spending'].append(sim_dict['spending'])
        result_pd['avg_acc'].append(sim_dict['avg_acc'])
    res_df = pd.DataFrame.from_dict(result_pd)
    res_df.to_csv(args.output,index=False)
    print(f"Simulation complete, total time elapsed: {time.perf_counter() - time_start:.3f}s")

# Execute the main simulation
main(args)

Trial 1/20... (Time elapsed: 11.097s)
Trial 2/20... (Time elapsed: 2.772s)
Trial 3/20... (Time elapsed: 2.819s)
Trial 4/20... (Time elapsed: 2.567s)
Trial 5/20... (Time elapsed: 7.246s)
Trial 6/20... (Time elapsed: 5.573s)
Trial 7/20... (Time elapsed: 3.336s)
Trial 8/20... (Time elapsed: 3.642s)
Trial 9/20... (Time elapsed: 3.376s)
Trial 10/20... (Time elapsed: 2.613s)
Trial 11/20... (Time elapsed: 3.329s)
Trial 12/20... (Time elapsed: 3.471s)
Trial 13/20... (Time elapsed: 3.301s)
Trial 14/20... (Time elapsed: 2.650s)
Trial 15/20... (Time elapsed: 3.067s)
Trial 16/20... (Time elapsed: 3.288s)
Trial 17/20... (Time elapsed: 2.525s)
Trial 18/20... (Time elapsed: 2.552s)
Trial 19/20... (Time elapsed: 2.566s)
Trial 20/20... (Time elapsed: 3.776s)
Simulation complete, total time elapsed: 75.688s
